# Modul 8: UnifoLM-VLA-0 — architektura i ewaluacja na LIBERO

**UnifoLM-VLA-0** to oficjalny model VLA (Vision-Language-Action) od **Unitree Robotics** — producenta robota humanoidalnego G1.

## Dlaczego UnifoLM?

- **Open-source**: pelny kod, wagi modelu i dane treningowe sa publicznie dostepne
- **Zaprojektowany dla G1**: dostarczany z 12 datasetami manipulacji na prawdziwym robocie G1
- **State-of-the-art**: na benchmarku LIBERO osiaga **98.7%** (vs GR00T N1: 93.9%)
- **Flow-matching**: uzywa DiT z flow-matching zamiast klasycznej dyfuzji, co daje plynniejsze ruchy

W tym notebooku przeanalizujemy architekture UnifoLM, porownamy go z innymi VLA, i zbadamy pipeline fine-tuningu.

In [ ]:
!pip install -q transformers torch matplotlib numpy pandas huggingface_hub

## Architektura UnifoLM-VLA-0

UnifoLM-VLA-0 to model **dual-system** — laczy duzy model VLM (rozumienie sceny) z szybka glowica akcji (generowanie ruchow):

```
Obraz z kamery + instrukcja tekstowa
         |
         v
┌─────────────────────────────────────┐
│  Qwen2.5-VL-7B (backbone VLM)      │
│  - Fine-tuned na VQA + dane robotow │
│  - Rozumienie sceny i instrukcji    │
│  - ~7 miliardow parametrow          │
│         |                           │
│         v last hidden state         │
│  DiT Action Head (flow-matching)    │
│  - Diffusion Transformer            │
│  - Generuje action chunks           │
│  - 4-8 krokow flow-matching         │
└──────────────┬──────────────────────┘
               |
               v
  Action chunks: joint positions [14 DOF]
  (sekwencja 5-20 przyszlych krokow)
               |
               v
  Robot G1: WBC / kontroler niskiego poziomu (200 Hz)
```

**Kluczowa roznica vs GR00T N1**: UnifoLM uzywa **flow-matching** (odmiana dyfuzji) zamiast deterministycznej glowicy akcji. Generuje **action chunks** — sekwencje kilku przyszlych krokow — co poprawia plynnosc ruchow.

## Porownanie z innymi VLA

Jak UnifoLM-VLA-0 wypada na tle konkurencji?

In [ ]:
import pandas as pd

data = {
    'Model': ['UnifoLM-VLA-0', 'GR00T N1', 'GR00T N1.6', 'OpenVLA', 'pi0'],
    'Producent': ['Unitree', 'NVIDIA', 'NVIDIA', 'Stanford', 'Physical Intelligence'],
    'Params': ['7B+', '2.2B', '~3B', '7B', '3B'],
    'Backbone': ['Qwen2.5-VL-7B', 'Eagle-2B', 'Eagle+', 'Llama-2-7B', 'PaliGemma-3B'],
    'Action Head': ['DiT flow-matching', 'Deterministyczny', 'Deterministyczny', 'Autoregresywny', 'DiT flow-matching'],
    'LIBERO avg': ['98.7%', '93.9%', 'N/A', '78.3%', 'N/A'],
    'Open Source': ['Tak (pelne)', 'Czesciowo', 'Czesciowo', 'Tak', 'Nie'],
    'Datasety G1': [12, 0, 0, 0, 0],
}

df = pd.DataFrame(data)
print("=== Porownanie modeli VLA ===")
print()
print(df.to_string(index=False))
print()
print("UnifoLM-VLA-0 ma najlepszy wynik na LIBERO i jako jedyny")
print("dostarcza gotowe datasety do manipulacji na robocie G1.")

## Wagi modelu z HuggingFace

Unitree udostepnia wagi modelu na HuggingFace. Sprawdzmy metadane bez pobierania pelnych wag (~14GB).

In [ ]:
from huggingface_hub import model_info

models_to_check = [
    "unitreerobotics/UnifoLM-VLA-Base",
    "unitreerobotics/UnifoLM-VLA-Libero",
    "unitreerobotics/UnifoLM-VLM-Base",
]

for model_id in models_to_check:
    print(f"\n{'=' * 50}")
    try:
        info = model_info(model_id)
        print(f"Model: {info.modelId}")
        print(f"Downloads: {info.downloads}")
        print(f"Tags: {info.tags}")
        if info.safetensors:
            size_gb = info.safetensors.total / 1e9
            print(f"Size: {size_gb:.1f} GB")
        else:
            print("Size: unknown")
    except Exception as e:
        print(f"Model: {model_id}")
        print(f"Nie mozna pobrac info (wymagany dostep do HF): {e}")
        print("Architektura: Qwen2.5-VL-7B + DiT flow-matching action head")
        print("Wiecej: https://huggingface.co/unitreerobotics")

## Pipeline fine-tuningu

Fine-tuning UnifoLM-VLA wymaga:
- **CUDA 12.4** + FlashAttention2
- **~40GB VRAM** (RTX 4090 lub A100)
- Dane w formacie **RLDS** (konwersja z LeRobot)

Ponizej kluczowe parametry i komenda treningowa.

In [ ]:
# Komenda treningowa UnifoLM-VLA (NIE uruchamiaj w Colab — wymaga duzego GPU)

training_command = """
# =================================================================
# Fine-tuning UnifoLM-VLA-0 na wlasnych danych G1
# =================================================================
# Wymagania: CUDA 12.4, FlashAttention2, ~40GB VRAM
# Base model: UnifoLM-VLA-Base z HuggingFace

accelerate launch --num_processes 1 \\
  unifolm_vla/train.py \\
  --pretrained_path unitreerobotics/UnifoLM-VLA-Base \\
  --dataset_name my_g1_demos \\
  --batch_size 4 \\
  --learning_rate 2e-5 \\
  --num_epochs 50 \\
  --action_chunk_size 10 \\
  --flow_matching_steps 4 \\
  --output_dir ./checkpoints/my_g1_vla
"""

print(training_command)

print("Parametry do eksperymentowania:")
print("="  * 50)
params = {
    "--batch_size": "2-8 (wiecej = szybciej, ale wiecej VRAM)",
    "--flow_matching_steps": "4-8 (wiecej = dokladniej, wolniej)",
    "--action_chunk_size": "5-20 (dluzsze chunki = plynniejsze ruchy)",
    "--learning_rate": "1e-5 do 5e-5",
    "--num_epochs": "30-100 (zalezne od ilosci danych)",
}
for param, desc in params.items():
    print(f"  {param:<25} {desc}")

## Format danych: RLDS

UnifoLM wymaga danych w formacie **RLDS** (Reinforcement Learning Datasets), nie LeRobot. Ponizej pipeline konwersji.

In [ ]:
# Pipeline konwersji danych: LeRobot → HDF5 → RLDS
# (ten kod pokazuje strukture — NIE uruchamiaj bezposrednio)

conversion_pipeline = """
# =================================================================
# Krok 1: LeRobot → HDF5
# =================================================================
# Konwertuje dane z formatu LeRobot v3.0 do HDF5
# Wejscie: katalog z epizodami LeRobot (parquet + video)
# Wyjscie: pliki HDF5 z obrazami i akcjami

python scripts/convert_lerobot_to_hdf5.py \\
  --lerobot_path ./data/my_g1_demos \\
  --output_path ./data/hdf5/my_g1_demos

# =================================================================
# Krok 2: HDF5 → RLDS
# =================================================================
# Konwertuje HDF5 do formatu RLDS (tfrecord)
# RLDS to standardowy format dla robotycznych datasetow (Open X-Embodiment)

python scripts/convert_hdf5_to_rlds.py \\
  --hdf5_path ./data/hdf5/my_g1_demos \\
  --output_path ./data/rlds/my_g1_demos

# =================================================================
# Krok 3: Rejestracja datasetu
# =================================================================
# Dodaj konfiguracje w unifolm_vla/configs/constants.py:

MY_G1_DATASET = {
    "action_dim": 14,         # DOF ramion G1
    "state_dim": 14,          # proprioceptive state
    "action_chunk_size": 10,  # predykcje w jednym kroku
    "camera_keys": ["cam_high", "cam_wrist"],
    "action_norm": "min_max",
}
"""

print(conversion_pipeline)
print("Uwaga: Skrypty konwersji dostepne w repozytorium unifolm-vla")
print("https://github.com/unitreerobotics/unifolm-vla")

## Kiedy UnifoLM vs GR00T?

Wybor miedzy UnifoLM-VLA-0 a GR00T N1.6 zalezy od Twojego scenariusza.

In [ ]:
decision_tree = """
=================================================================
          DRZEWO DECYZYJNE: UnifoLM vs GR00T
=================================================================

Czy pracujesz z robotem Unitree G1?
│
├── TAK
│   │
│   ├── Czy masz GPU z >= 24GB VRAM?
│   │   │
│   │   ├── TAK → UnifoLM-VLA-0
│   │   │   - 12 gotowych datasetow G1
│   │   │   - LIBERO 98.7%
│   │   │   - Pelne open-source
│   │   │
│   │   └── NIE → GR00T N1.6
│   │       - Mniejszy model (2.2B vs 7B)
│   │       - Szybszy fine-tuning (~2000 steps)
│   │       - Wymaga mniej VRAM
│   │
│   └── Chcesz oba? → GR00T do prototypowania, UnifoLM do produkcji
│
└── NIE (inny robot)
    │
    ├── Czy masz wsparcie Isaac-GR00T?
    │   │
    │   ├── TAK → GR00T N1.6
    │   │   - Cross-platform deployment
    │   │   - NVIDIA ekosystem
    │   │
    │   └── NIE → OpenVLA lub pi0
    │       - OpenVLA: pelne open-source, 7B
    │       - pi0: zamkniety, ale potezny
    │
    └── Badania akademickie → OpenVLA (najbardziej otwarty)

=================================================================
PODSUMOWANIE:
  UnifoLM = najlepszy na G1, wymaga wiecej VRAM
  GR00T   = szybszy, mniejszy, cross-platform
  OpenVLA = najlepszy do badan, ale slabsze wyniki
=================================================================
"""

print(decision_tree)

## Wnioski

### UnifoLM-VLA-0 — najlepszy open-source VLA dla G1

**Zalety:**
- Najlepszy wynik na LIBERO benchmark (98.7%) — pokonuje GR00T N1 (93.9%) i OpenVLA (78.3%)
- Pelne open-source: kod, wagi i dane treningowe
- 12 gotowych datasetow manipulacji na robocie G1
- Flow-matching (DiT) daje plynniejsze ruchy niz deterministyczny GR00T

**Ograniczenia:**
- **Wiecej VRAM**: Qwen2.5-VL-7B wymaga ~40GB do treningu (vs ~16GB dla GR00T N1)
- **Wolniejszy fine-tuning**: wiekszy model = wiecej epok, wiecej czasu
- **Tylko G1**: datasety i konfiguracja zoptymalizowane pod Unitree G1 (GR00T jest cross-platform)
- **Format RLDS**: wymaga konwersji z LeRobot (dodatkowy krok w pipeline)

### Rekomendacja

| Scenariusz | Rekomendacja |
|-----------|-------------|
| Praca z G1 + duzy GPU | **UnifoLM-VLA-0** |
| Szybki prototyp na G1 | **GR00T N1.6** |
| Inny robot / cross-platform | **GR00T N1.6** |
| Badania akademickie | **OpenVLA** |
| Produkcja na G1 | **GR00T → prototyp, UnifoLM → produkcja** |